# Текущий контроль 1 — решение
## Парная линейная регрессия для гиперспектра кукурузы
Исходные задания взяты из «Текущий контроль 1.ipynb», данные — из «Гиперспектр кукурузы.csv».
Здесь выполнены три задания по CSV и построена кривая изменения ошибки.
Обозначения: $x$ — `wavelength`, $Y$ — `Spectr`. Единицы измерения в CSV не указаны.

Для повторного запуска положите CSV рядом с ноутбуком и выполните ячейки сверху вниз. Нужны NumPy, pandas и Matplotlib.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

plt.rcParams.update({'figure.figsize': (10, 5), 'font.size': 11})
gipers = pd.read_csv('Гиперспектр кукурузы.csv', sep=';', decimal=',')
display(gipers.head())
print('Размер таблицы:', gipers.shape)
print('Пропуски:', gipers.isna().sum().to_dict())
x = gipers['wavelength'].to_numpy(dtype=float)
Y = gipers['Spectr'].to_numpy(dtype=float)
assert np.isfinite(x).all() and np.isfinite(Y).all()
assert len(x) > 1 and np.var(x) > 0
print(f'Диапазон x: {x.min():.2f} — {x.max():.2f}')
print(f'Диапазон Y: {Y.min():.4f} — {Y.max():.4f}')

## 1. Аналитическое решение
Ищем прямую $\widehat Y=a_0+a_1x$, минимизирующую средний квадрат ошибки:
$$MSE=\frac1n\sum_{i=1}^{n}(a_0+a_1x_i-Y_i)^2.$$
Приравнивая производные к нулю, получаем:
$$a_1=\frac{\sum_i(x_i-\bar x)(Y_i-\bar Y)}{\sum_i(x_i-\bar x)^2},\qquad a_0=\bar Y-a_1\bar x.$$


In [ ]:
a1 = np.sum((x - x.mean()) * (Y - Y.mean())) / np.sum((x - x.mean()) ** 2)
a0 = Y.mean() - a1 * x.mean()
Y_analytic = a0 + a1 * x
print(f'Аналитическая модель: Y_hat = {a0:.10f} + {a1:.10f} * x')
order = np.argsort(x)
plt.figure()
plt.scatter(x, Y, s=14, alpha=0.7, label='Исходные данные')
plt.plot(x[order], Y_analytic[order], color='crimson', label='Аналитическая регрессия')
plt.xlabel('Длина волны (wavelength)')
plt.ylabel('Спектральный показатель (Spectr)')
plt.title('Аналитическое решение')
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 2. Градиентный спуск
Чтобы большие значения длины волны не вызывали расходимость, используем $z=(x-\bar x)/s_x$ и модель $\widehat Y=b_0+b_1z$.
Производные **MSE**:
$$g_0=2\operatorname{mean}(\widehat Y-Y),\qquad g_1=2\operatorname{mean}((\widehat Y-Y)z).$$
Оба коэффициента обновляем одновременно: $b_j\leftarrow b_j-\alpha g_j$.
Останавливаемся при малой норме градиента или достижении лимита шагов.
Коэффициенты исходной модели: $a_1=b_1/s_x$, $a_0=b_0-a_1\bar x$.

В учебном примере используются суммы, то есть градиент SSE, а не MSE: при одинаковом `alpha` шаг больше в $n$ раз. Условие `MSE < epsylon` также может быть недостижимым. В решении ниже критерий остановки проверяет близость к минимуму, а не заранее заданную величину ошибки.


In [ ]:
class SimpleRegression:
    def fit(self, x, Y, alpha=0.1, tol=1e-10, max_steps=10000):
        self.x_mean = x.mean()
        self.x_std = x.std()
        z = (x - self.x_mean) / self.x_std
        b0, b1 = 0.0, 0.0
        self.errors = [float(np.mean(Y ** 2))]
        self.converged = False
        for _ in range(max_steps):
            error = b0 + b1 * z - Y
            g0 = 2 * error.mean()
            g1 = 2 * (error * z).mean()
            if np.hypot(g0, g1) < tol:
                self.converged = True
                break
            b0, b1 = b0 - alpha * g0, b1 - alpha * g1
            self.errors.append(float(np.mean((b0 + b1 * z - Y) ** 2)))
        self.a1 = b1 / self.x_std
        self.a0 = b0 - self.a1 * self.x_mean
        self.n_steps = len(self.errors) - 1
        return self

    def predict(self, x):
        return self.a0 + self.a1 * x

regr = SimpleRegression().fit(x, Y)
Y_gd = regr.predict(x)
print(f'Градиентный спуск: Y_hat = {regr.a0:.10f} + {regr.a1:.10f} * x')
print('Шагов:', regr.n_steps, '| Критерий сходимости выполнен:', regr.converged)
print('Максимальное различие прогнозов:', np.max(np.abs(Y_gd - Y_analytic)))
assert regr.converged
assert np.allclose(Y_analytic, Y_gd, rtol=0, atol=1e-8)
# Независимая проверка аналитической формулы методом наименьших квадратов.
check = np.linalg.lstsq(np.column_stack([np.ones_like(x), x]), Y, rcond=None)[0]
assert np.allclose([a0, a1], check)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(x, Y, s=12, alpha=0.7, label='Исходные данные')
axes[0].plot(x[order], Y_gd[order], color='darkorange', label='Градиентный спуск')
axes[0].set(xlabel='wavelength', ylabel='Spectr', title='Модель после обучения')
axes[0].legend()
axes[1].plot(range(len(regr.errors)), regr.errors)
axes[1].axhline(np.mean((Y_analytic - Y)**2), color='crimson', linestyle='--', label='Минимум MSE (аналитически)')
axes[1].set(xlabel='Число обновлений', ylabel='MSE', title='Изменение ошибки при обучении')
axes[1].legend()
for ax in axes:
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 3. Оценка качества и сравнение
Вычислим MSE, RMSE, MAE и $R^2=1-\frac{\sum(Y_i-\widehat Y_i)^2}{\sum(Y_i-\bar Y)^2}$.
RMSE и MAE выражены в единицах `Spectr`, MSE — в квадрате этих единиц.
Все показатели рассчитаны на обучающих данных, как требуется в задании. Они описывают подгонку к данному спектру, но не оценивают качество на новых образцах.


In [ ]:
def metrics(pred):
    e = Y - pred
    mse = np.mean(e ** 2)
    return {'MSE': mse, 'RMSE': np.sqrt(mse), 'MAE': np.mean(np.abs(e)),
            'R²': 1 - np.sum(e ** 2) / np.sum((Y - Y.mean()) ** 2)}

results = pd.DataFrame([metrics(Y_analytic), metrics(Y_gd)],
                       index=['Аналитический метод', 'Градиентный спуск'])
display(results)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(x, Y, s=12, alpha=0.7, label='Исходные данные')
axes[0].plot(x[order], Y_analytic[order], color='crimson', linewidth=3, label='Аналитический метод')
axes[0].plot(x[order], Y_gd[order], color='black', linestyle='--', label='Градиентный спуск')
axes[0].set(xlabel='wavelength', ylabel='Spectr', title='Сравнение моделей')
axes[0].legend()
axes[1].scatter(x, Y - Y_analytic, s=12)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set(xlabel='wavelength', ylabel='Y − Y_hat', title='Остатки линейной модели')
for ax in axes:
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()
print(f'Доля объяснённой вариации: {results.iloc[0]["R²"]:.2%}')

## Вывод
По 273 наблюдениям получена модель:

**Ŷ = -0.18342783 + 0.00066816 · x**.

Аналитический метод и градиентный спуск дают практически одинаковые коэффициенты и прогнозы. Оба минимизируют одну и ту же квадратичную функцию ошибки.
MSE = **0.01768411**, RMSE = **0.132982**, R² = **0.435891**.
Линейная модель объясняет около **43.59%** вариации `Spectr`. Совпадение двух прямых подтверждает согласованность методов, но само по себе не доказывает хорошее описание спектра. Форму отклонений оцениваем по графику остатков.


На графике остатков наблюдаются систематические изгибы: в области x ≈ 750–880 модель преимущественно занижает Spectr, а около 920–960 — завышает. Поэтому прямая передаёт общий рост, но не описывает локальные пики и провалы спектра. Градиентный спуск сошёлся за 102 шага, максимальное различие прогнозов двух методов меньше 7·10⁻¹¹.
